# 002 Subagents Handoffs

这是 LangChain Multi-agent 学习线的第二份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/multi-agent/subagents

学习目标：

1. 理解 subagents 架构：main agent / supervisor 通过 tools 调用子 agent
2. 理解 subagent 默认无状态，主要价值是上下文隔离
3. 跑通 tool-per-agent 的最小实现
4. 理解 single dispatch tool 的注册表模式
5. 区分 sync / async subagent 执行取舍
6. 学习 subagent specs、inputs、outputs 的上下文工程
7. 对比 subagent 和 handoff 的控制权差异

这一讲使用 fake model，不消耗真实模型额度。

## 1. Subagents 的核心架构

Subagents 模式里有一个 central main agent，也叫 supervisor。

它负责：

- 维护主对话上下文
- 决定调用哪个 subagent
- 给 subagent 提供任务输入
- 接收 subagent 结果
- 综合结果并继续对话

subagent 不直接和用户对话，它被包装成 tool，由 supervisor 调用。

```text
user
  -> supervisor/main agent
      -> tool: research_subagent(query)
          -> research agent 独立上下文执行
      <- research result
  <- supervisor synthesis
```

这和 handoff 不一样。handoff 是 active agent 发生切换；subagent 是 supervisor 始终掌握控制权。

In [20]:
from enum import Enum

from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeListChatModel, FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.tools import tool


class ToolCallingFakeModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self


def print_messages(result: dict) -> None:
    for message in result.get("messages", []):
        print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)


## 2. Tool per agent：把 subagent 包成工具

最直接的实现方式是：

1. 创建一个 subagent
2. 写一个 tool 函数调用这个 subagent
3. 把这个 tool 注册给 main agent

main agent 看到的是一个普通 tool，但这个 tool 内部其实启动了一个 agent。

In [21]:
research_agent = create_agent(
    model=FakeListChatModel(responses=["research result: 找到 3 条相关事实。"]),
    tools=[],
)


@tool("research", description="Research a topic and return concise findings.")
def call_research_agent(query: str) -> str:
    """Call the research subagent."""
    result = research_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content


main_model = ToolCallingFakeModel(
    responses=[
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "research",
                    "args": {"query": "调查 LangChain subagents 的核心特点"},
                    "id": "call_1",
                }
            ],
        ),
        AIMessage(content="综合结果：subagent 通过 tool 被 supervisor 调用，并返回简洁观察。"),
    ]
)

main_agent = create_agent(
    model=main_model,
    tools=[call_research_agent],
)

result = main_agent.invoke({"messages": [{"role": "user", "content": "解释 subagents"}]})
print_messages(result)


human 解释 subagents
ai 
tool_calls: [{'name': 'research', 'args': {'query': '调查 LangChain subagents 的核心特点'}, 'id': 'call_1', 'type': 'tool_call'}]
tool research result: 找到 3 条相关事实。
ai 综合结果：subagent 通过 tool 被 supervisor 调用，并返回简洁观察。


## 3. Subagent 默认无状态

官方文档强调：subagents 是 stateless 的。

意思是：每次调用 subagent，通常只给它本次任务需要的输入，不把主对话完整历史都塞进去。

这样做的价值是上下文隔离。

In [22]:
isolated_agent = create_agent(
    model=FakeListChatModel(responses=["subagent only saw a clean task input."]),
    tools=[],
)


@tool("isolated_research", description="Run isolated research with only the task query.")
def isolated_research(query: str) -> str:
    subagent_input = {"messages": [{"role": "user", "content": query}]}
    print("subagent message count:", len(subagent_input["messages"]))
    result = isolated_agent.invoke(subagent_input)
    return result["messages"][-1].content


print(isolated_research.invoke({"query": "只调查 approval 恢复流程"}))


subagent message count: 1
subagent only saw a clean task input.


## 4. Single dispatch tool：一个 task 工具调多个 subagent

如果 subagent 很多，每个都写一个 tool 会变得重复。

另一种方式是维护一个 registry，用一个 `task(agent_name, description)` 工具分发。

适合：

- subagent 很多
- 不同团队维护不同 agent
- 希望用统一协议调用 subagent
- 更看重上下文隔离而不是每个 tool 的精细 schema

In [23]:
research_subagent = create_agent(
    model=FakeListChatModel(responses=["research: 找到了背景资料。"]),
    tools=[],
)

writer_subagent = create_agent(
    model=FakeListChatModel(responses=["writer: 已整理成清晰说明。"]),
    tools=[],
)

SUBAGENTS = {
    "research": research_subagent,
    "writer": writer_subagent,
}


@tool
def task(agent_name: str, description: str) -> str:
    """Launch an ephemeral subagent for a task. Available: research, writer."""
    agent = SUBAGENTS[agent_name]
    result = agent.invoke({"messages": [{"role": "user", "content": description}]})
    return result["messages"][-1].content


print(task.invoke({"agent_name": "research", "description": "调查 subagent 特点"}))
print(task.invoke({"agent_name": "writer", "description": "整理成中文说明"}))


research: 找到了背景资料。
writer: 已整理成清晰说明。


## 5. Enum constraint：让 dispatch 更类型安全

如果 registry 较小而且稳定，可以用 enum 约束 `agent_name`。

这样模型不能随便编造 agent 名字。

In [24]:
class AgentName(str, Enum):
    RESEARCH = "research"
    WRITER = "writer"


@tool
def typed_task(agent_name: AgentName, description: str) -> str:
    """Launch a registered subagent with an enum-constrained name."""
    agent = SUBAGENTS[agent_name.value]
    result = agent.invoke({"messages": [{"role": "user", "content": description}]})
    return result["messages"][-1].content


print(typed_task.invoke({"agent_name": "research", "description": "调查 enum 约束"}))


research: 找到了背景资料。


## 6. Sync vs Async：同步等待还是后台任务

这里的 async 不是简单说 Python `async/await`。

官方文档里的意思是：main agent 是否等待 subagent 完成。

| 模式 | main agent 行为 | 适合场景 |
| --- | --- | --- |
| sync | 等 subagent 完成 | 后续回答依赖 subagent 结果 |
| async/background | 启动后台任务后继续 | 子任务独立，用户不应该等待 |

后台任务通常需要三类工具：

1. start job
2. check status
3. get result

In [25]:
JOB_STORE: dict[str, dict] = {}


def start_background_research(description: str) -> str:
    job_id = "job_001"
    JOB_STORE[job_id] = {"status": "completed", "result": "后台 research 已完成：" + description}
    return job_id


def check_job_status(job_id: str) -> str:
    return JOB_STORE[job_id]["status"]


def get_job_result(job_id: str) -> str:
    return JOB_STORE[job_id]["result"]


job_id = start_background_research("调查 subagents async 设计")
print("job_id:", job_id)
print("status:", check_job_status(job_id))
print("result:", get_job_result(job_id))


job_id: job_001
status: completed
result: 后台 research 已完成：调查 subagents async 设计


## 7. Context engineering：specs、inputs、outputs

Subagent 设计最容易出问题的地方是上下文。

官方文档把它拆成三类：

| 类别 | 目的 |
| --- | --- |
| subagent specs | 让 main agent 知道什么时候调用哪个 subagent |
| subagent inputs | 让 subagent 拿到刚好足够的上下文 |
| subagent outputs | 让 supervisor 拿到可综合的结果 |

这和我们之前说的 synthesis 完全一致：subagent 不应该返回一堆原始细节，而要返回主流程能用的观察。

In [26]:
subagent_specs = [
    {
        "name": "research",
        "description": "Use for read-only fact finding and source inspection.",
    },
    {
        "name": "writer",
        "description": "Use for turning verified facts into clear Chinese explanations.",
    },
]


def format_subagent_result(agent_name: str, result: str, confidence: str) -> str:
    return f"agent={agent_name}\nconfidence={confidence}\nsummary={result}"


print(format_subagent_result("research", "找到 subagents 的核心机制是 tools 调用。", "high"))


agent=research
confidence=high
summary=找到 subagents 的核心机制是 tools 调用。


## 8. Subagent vs Handoff

这节课主体是 subagents，但要先建立和 handoff 的区别。

| 对比 | Subagent | Handoff |
| --- | --- | --- |
| 控制权 | supervisor 始终控制 | active agent 发生切换 |
| 用户交互 | subagent 通常不直接面向用户 | 新 agent 可以继续和用户对话 |
| 上下文 | 子任务隔离，返回结果 | 接管对话上下文 |
| 典型用法 | research、verification、局部任务 | 售前转合同、客服转技术支持 |

下一讲如果继续，就可以专门学习 handoffs。

## 9. 和本仓库 Harness 的对应关系

| LangChain Subagents | 本仓库 Harness |
| --- | --- |
| supervisor main agent | `HarnessChatAgent` / coordinator |
| subagent as tool | `delegate` 后进入 subagent 执行单元 |
| stateless subagent | 子 agent 默认隔离上下文 |
| subagent result | `subagent_result` / synthesis 输入 |
| sync subagent | 当前 query loop 等待结果 |
| async subagent | 后台任务 + job status + resume/summarize |
| specs / inputs / outputs | subagent prompt、allowed_tools、allowed_paths、summary contract |

关键判断：

```text
subagent 不是另一个聊天窗口，而是一个受 supervisor 调用的隔离执行单元。
```

## 10. 本讲练习

请判断下面设计更适合 tool-per-agent 还是 single dispatch tool：

1. 只有 research、writer、reviewer 三个稳定 subagent。
2. 企业里有几十个团队，各自维护自己的 subagent。
3. 每个 subagent 的输入 schema 差异很大。
4. 你希望新增 subagent 时不改 coordinator 代码。

参考答案：

1. tool-per-agent 或 enum dispatch 都可以
2. single dispatch tool
3. tool-per-agent
4. single dispatch tool

## 11. 本讲小结

这一讲的核心：

```text
Subagent 模式 = supervisor 保持控制权 + subagent 作为 tool 被调用 + 上下文隔离 + 结果返回后由 supervisor synthesis。
```

你现在应该能判断：

- subagent 为什么默认无状态
- 什么时候用 tool-per-agent
- 什么时候用 single dispatch tool
- sync 和 async subagent 的区别
- subagent 和 handoff 的关键差异

下一讲可以继续学习 handoffs 或进入 skills。